#### スクレイピング

どのサイトを選ぶかで、得られる情報の質が決まる
どの情報を取るかで、分析の深さが変わる


#### PythonでWebスクレイピングを行う方法

In [ ]:
requests: HTMLデータを取得するライブラリ
BeautifulSoup: HTMLやXMLの解析を行うライブラリ
Scrapy: より高度で大規模なスクレイピングに適したフレームワーク
Selenium: JavaScriptコンテンツを含む動的ページも対応

（1）WebページにアクセスしてHTMLデータを取得

In [1]:
import requests
url = "https://example.com"
response = requests.get(url)
html = response.text

☆なんだここ!?　IANAて機関があるのね

（2）HTMLの解析と必要な情報の抽出

In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
title = soup.find("title").text

（3）データの保存

In [4]:
import csv
with open("data.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Title"])
    writer.writerow([title])

（4）エラー処理と例外対応

In [ ]:
エラーにもちゃんと対応を。

#### 言語モデルを活用したWebスクレイピング

以前のスクレイピングでは、HTML構造への依存が高く
構造が少しでも変わるとコードもメンテ必要。柔軟性も低い

LLM利用のスクレイピングでは自然言語での指示で柔軟に情報を
取得できる。

一方で、API利用料（コスト）は高く、処理速度は遅く、指示によって情報のばらつきも生じる特徴がある。

In [6]:
#従来のスクレイピング

import requests
from bs4 import BeautifulSoup

# ニュース一覧ページのURL
url = "https://www.kiramex.com/news/"

# ページのHTMLを取得
response = requests.get(url)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパース（意味のある情報だけを抜き出す）
# BeautifulSoupは、HTMLの解析と要素の抽出するライブラリ。
soup = BeautifulSoup(response.text, "html.parser")

# ニュース一覧の情報を抽出
news_list = soup.find_all("li", class_="news-block")

# ニュース情報の取得
for news in news_list:
    # ニュースの日付を取得
    date = (
        news.find("p", class_="date").get_text(strip=True) 
        if news.find("p", class_="date") 
        else "N/A"
    )
    
    # ニュースのタイトルを取得
    title = news.find("a", class_="title").get_text(strip=True)
    
    # ニュースのリンクを取得
    link = news.find("a", class_="title")["href"]

    print(f"Date: {date}")
    print(f"Title: {title}")
    print(f"Link: {link}")
    print("-" * 40)

Date: 2025年04月01日お知らせ
Title: コーポレートサイトリニューアルお知らせ
Link: https://www.kiramex.com/news-20250401/
----------------------------------------
Date: 2025年02月06日メディア掲載
Title: LiProにテックアカデミーが紹介されました
Link: https://www.kiramex.com/news-20250206/
----------------------------------------
Date: 2025年01月14日プレスリリース
Title: リスキリングプログラム「LINEヤフーテックアカデミー」、2週間でChatGPTの基礎や使い方を学べる「はじめてのChatGPTコース」など4種の新コースを開設
Link: https://www.kiramex.com/news-20250114/
----------------------------------------
Date: 2024年12月01日お知らせ
Title: 年末年始（2024年〜2025年）の休暇・営業に関するお知らせ
Link: https://www.kiramex.com/news-20241201/
----------------------------------------
Date: 2024年10月09日プレスリリース
Title: キラメックスとLINEヤフー、山形県の産学官連携コンソーシアム「やまがたAI部」と、県内企業におけるAI人材育成に関する協定を締結
Link: https://www.kiramex.com/news-20241009/
----------------------------------------
Date: 2024年09月09日プレスリリース
Title: テックアカデミー、学びに夢中になれる「イマーシブラーニング」を取り入れたコースの提供を開始　〜副業をスタートした受講者が3倍に増加した実証実験の実績〜
Link: https://www.kiramex.com/news-20240909/
----------------------------------------
D

In [ ]:
#従言語モデルを活用したスクレイピング

# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアント生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


In [ ]:

#言語モデルに与えるためのHTMLを取得

# ニュース一覧ページのURL
url = "https://www.kiramex.com/news/"

# ページのHTMLを取得
response = requests.get(url)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパースし、body部分を取り出す
soup = BeautifulSoup(response.text, "html.parser")
body_html = str(soup.body)  # body部分のHTMLを文字列として取得
print(body_html) # 結果を表示して確認


<body id="top">
<header>
<div class="inner">
<div class="logo"><a href="/"><img alt="キラメックス株式会社" src="https://www.kiramex.com/wp-content/themes/kiramex/images/logo.png"/></a></div>
<nav class="navbar navbar-default" role="navigation">
<div class="container-fluid">
<div class="navbar-header"><button class="navbar-toggle" data-target="#header-menu" data-toggle="collapse" type="button"><span class="sr-only">Toggle navigation</span><span class="icon-bar"></span><span class="icon-bar"></span><span class="icon-bar"></span></button></div>
<div class="collapse navbar-collapse" id="header-menu">
<ul class="nav navbar-nav">
<li><a class="" href="/company/">会社情報</a></li>
<li><a class="" href="/service/">事業情報</a></li>
<li><a class="" href="/news/">ニュース</a></li>
<li><a class="" href="/contact/">お問い合わせ</a></li>
<li class="recruit"><a class="" href="/recruit/">採用情報</a></li>
</ul>
</div>
</div>
</nav>
</div>
</header><div class="cover-news" id="cover">
<div class="inner">
<h1>ニュース</h1>
</div>
</div>
<

↑　”意味のある情報だけ”が取得できるbeautifulsoupが効いてる

In [ ]:
# LLMにニュース一覧を抽出させるプロンプトを作成
prompt = f"""
以下のHTMLから最新のニュースを抽出し、「日付、タイトル、リンク」の形式で一覧を出力してください。一覧以外は出力しないでください。

# 出力様式：
Date: 日付
Title: タイトル
Link: リンク
--------------------

#HTML:
{body_html[:5000]}
"""
# ↑  API利用料を抑えるために、「入力」トークン長を制限


# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500, # こっちは「出力」トークン長を制限
    temperature=0.3 # 回答の確実性を高め、ランダム性を抑える
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

```
Date: 2025年04月01日
Title: コーポレートサイトリニューアルお知らせ
Link: https://www.kiramex.com/news-20250401/
--------------------
Date: 2025年02月06日
Title: LiProにテックアカデミーが紹介されました
Link: https://www.kiramex.com/news-20250206/
--------------------
Date: 2025年01月14日
Title: リスキリングプログラム「LINEヤフーテックアカデミー」、2週間でChatGPTの基礎や使い方を学べる「はじめてのChatGPTコース」など4種の新コースを開設
Link: https://www.kiramex.com/news-20250114/
--------------------
```


★ トークン長は、これが妥当な長さ、の判断材料は？ (AIに聞く？

#### LLMを活用したWebスクレイピングの活用アイデア
- 製品・サービスの競合比較表作成　(企業ごとの価格表とか)
- 要約やトレンド分析の自動化　（特定業種の経済動向とか）
- 質問応答システムの構築　（チャットボット系）
- 高度なテキスト解析と分類　（マーケ分野での感情分析とか）
- ECサイトの自動レビュー生成　（典型的なレビュー生成とか）
- 競合分析と自動レポート生成　（競合企業の迅速な動向調査）
- 定期レポートの自動生成と要約　(フォームの決まったマーケットレポートとか)

#### APIを利用するメリット

APIが提供されている場合は、スクレイピングではなくAPIを利用することが望ましい　公式データで安心、信頼できる、安定する。

例：X(ツイッター)、Yahoo finance、GoogleMaps、OpenWeather(天気)、
ほか、調べるといろいろ
OpenCorporates(登記情報)、厚生労働省、経済産業省、国税庁、Udemy for Business API、 pytrends(使う前には調査・確認を)　etc。


##### - Wikipedia　API

https://ja.wikipedia.org/w/api.php にリクエストを送る。

https://www.mediawiki.org/wiki/API:Action_API 詳細こちら

主なパラメータ：

- action: 実行したいアクションの種類を指定します。一般的なものはquery（情報の取得）とsearch（検索）です。
- format: 返されるデータの形式を指定します。一般的にjsonが使用されます。
- titles: 特定のページを指定するときに使用します。
- list: 検索やカテゴリなど、特定のリストを取得するときに使用します。-


In [ ]:
# Wikipedia API

import requests

# Wikipedia日本語版のAPIエンドポイント
WIKI_API_URL = "https://ja.wikipedia.org/w/api.php"

# 共通のヘッダー（User-Agentを設定する） 
HEADERS = { "User-Agent": "MyWikipediaClient/1.0 (my_email@example.com)" }



# 検索キーワードに該当するWikipediaページを10件取得する関数
def fetch_search_results(keyword, limit=10):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "format": "json",
        "srlimit": limit
    }

    try:
        response = requests.get(WIKI_API_URL, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        return data.get("query", {}).get("search", [])
    except requests.RequestException as e:
        print("検索リクエストでエラーが発生しました:", e)
        return []

# 指定されたタイトルのページ内容を取得する関数
def fetch_page_content(title):
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True
    }

    try:
        response = requests.get(WIKI_API_URL, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        page = next(iter(data.get("query", {}).get("pages", {}).values()), {})
        return page.get("extract", "ページ内容が見つかりません。")
    except requests.RequestException as e:
        print("ページ内容取得でエラーが発生しました:", e)
        return "ページ内容が取得できませんでした。"


In [24]:
# テストコードで確認
keyword = "Python"

# キーワードに該当するページ一覧を取得
search_results = fetch_search_results(keyword)
if search_results:
    print("【キーワードに該当するページ一覧】")
    for result in search_results[:3]:
        print(f"- {result['title']}")

    # 最初のページタイトルを取得
    first_title = search_results[0]["title"]
    print("\n【最初のページタイトル】")
    print(first_title)

    # 最初のページの内容を取得
    page_content = fetch_page_content(first_title)
    print("\n【ページ内容】")
    print(page_content[:500])
else:
    print("検索結果が見つかりません。")

【キーワードに該当するページ一覧】
- Python
- モンティ・パイソン
- IronPython

【最初のページタイトル】
Python

【ページ内容】
Python（パイソン）は、主としてインタープリタ的に実行される高水準汎用プログラミング言語である。


== 概要 ==
Pythonは1990年代初頭にグイド・ヴァン・ロッサムが開発し、1991年2月に0.9.0として公開された。
最初にリリースされたPythonの設計哲学は、ホワイトスペース（オフサイドルール）の顕著な使用によってコードの可読性を重視している。その言語構成とオブジェクト指向のアプローチは、プログラマが小規模なプロジェクトから大規模なプロジェクトまで、明確で論理的なコードを書くのを支援することを目的としている。
Pythonは動的言語で、動的型付き言語であり、ガベージコレクションがある。構造化（特に手続き型）、オブジェクト指向、関数型プログラミングを含む複数のプログラミングパラダイムをサポートしている。Pythonは、その包括的な標準ライブラリのため、しばしば「バッテリーを含む」言語と表現されている。
Pythonのインタプリタは多くのOSに対応している。プログラマーのグローバルコミュニティは、自由かつオープンソース  のリファレンス実装であるCPythonを開発お


In [27]:
# 得られたページ内容を、言語モデルに要約してもらう。

# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


# 要約を行うプロンプトを作成
prompt = f"""
以下の文章を要約してください。

# 条件：
- 小学生にもわかるように
- 300文字程度

# 文章：
{page_content[:1000]}
"""

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.3
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

Python（パイソン）は、プログラミングをするための言語の一つで、1990年代にグイド・ヴァン・ロッサムによって作られました。Pythonは、コードが読みやすく、理解しやすいことを大切にしています。この言語は、さまざまなプログラミングのスタイルに対応していて、小さなプロジェクトから大きなプロジェクトまで使えます。

Pythonは「インタプリタ」と呼ばれる方法で実行され、動的型付けやガベージコレクションといった特徴があります。これにより、プログラマーは簡単にコードを書くことができ、必要な機能をインターネットから追加することもできます。Pythonのコミュニティでは、シンプルで効率的なコードを書くことが重視されています。

このように、Pythonは使いやすく、たくさんの人に愛されているプログラミング言語です。
